# Decision Trees

## What will you learn in this course? 🧐🧐

This lecture is about giving you theoretical principles of decision trees, which allow you to establish successive binary rules to classify observations or to make regressions. Specifically, you will:

* Understand how **Decision Trees** create sequential, binary rules to predict a target variable.
* Differentiate between **Regression Trees** and **Classification Trees**.
* Explain how the tree is built: splitting rules, admissibility, heterogeneity, and stopping criteria.
* Recognize the problem of **overfitting** and how to mitigate it.
* Understand how the concept of **bagging** leads naturally to the **Random Forest** algorithm.

## What Is a Decision Tree? 🌳

Decision Trees belong to the family of **recursive partitioning** methods, commonly referred to as **CART** — *Classification and Regression Trees*.
They can be used for:

* **Regression** when the target variable $Y$ is *quantitative* (e.g., predicting a price),
* **Classification** when $Y$ is *qualitative* (e.g., predicting a label or category).

A decision tree breaks down the data step by step, applying simple binary rules at each step until a final prediction is made.
Its structure has three key elements:

* **Root** → contains all the training data (no split yet).
* **Nodes / Branches** → represent decision points that split the data based on a feature and a threshold.
* **Leaves** → terminal nodes that output a prediction (a numerical value for regression, a class for classification).

At each node, the algorithm asks a question like:

> “Is `X₁ < 5.2`?”
> and splits the data accordingly. This process repeats recursively, creating a hierarchy of rules.

A **stopping criterion** prevents the tree from growing indefinitely — we’ll come back to that shortly.

![decision\_tree](https://full-stack-assets.s3.eu-west-3.amazonaws.com/Decision+Tree.png)

Trees are appreciated because they’re **intuitive and interpretable**: you can visualize each rule and understand why a decision was made.


## How Is a Decision Tree Built? 🧩

Let’s go step by step.

### General Principle (Algorithm Outline)

1. **Initialization**

   * The root node starts with all training data.

2. **Splitting Step**

   * For each possible feature $X$, the algorithm tests all possible thresholds (for quantitative variables) or all possible groupings (for qualitative variables).
   * Each potential split is evaluated with respect to a **criterion** (e.g., variance reduction or Gini impurity).
   * The best split — the one that minimizes error or impurity — is chosen.

3. **Recursion**

   * The process repeats independently on each new branch (child node).

4. **Stopping**

   * The process stops when no more valid (admissible) splits are possible, or when certain conditions are met (e.g., minimum number of samples, maximum depth).

5. **Prediction Assignment**

   * Each **leaf node** is assigned a predicted value (regression) or a class (classification).

So, the tree grows *greedily*: it chooses the best split locally at each step, without necessarily finding the global optimum.

## Division Criteria ⚙️

### Admissibility

A split is **admissible** if:

* Both resulting branches contain at least one observation (no empty nodes).
* The variable allows for such a split.

If the variable $X$ is **quantitative**, there are $M-1$ possible split points for a node with $M$ observations.
If $X$ were nominal (categorical without order), there would be $2^{M-1} - 1$ possible splits — but in `scikit-learn`, categorical features must be **numerically encoded** beforehand.

### How Thresholds Are Chosen for Quantitative Variables 🧮

For quantitative variables, the tree doesn’t test every possible value.  
It follows a simple, systematic process:

1. **Sort** all unique values of $X$ in ascending order.  
2. **Compute midpoints** between consecutive values:
   $$
   t_k = \frac{x_{(i)} + x_{(i+1)}}{2}
   $$
   Each midpoint $t_k$ becomes a **candidate threshold**.  
3. For each threshold $t_k$, split the data into:
   - Left: $X \leq t_k$
   - Right: $X > t_k$
4. Compute the *impurity* for both sides (more on that in the next section).  
5. Choose the threshold that **minimizes the total impurity**:
   $$
   H_\text{left} + H_\text{right}
   $$

👉 Example: If $X = [2.0, 4.0, 5.5, 9.0]$, the possible thresholds are $t \in \{3.0, 4.75, 7.25\}$.

### Measuring Heterogeneity (or Impurity)

To choose the best split, the algorithm measures how *homogeneous* (pure) the resulting nodes are.
A good split **reduces heterogeneity** — it groups together observations that have similar values of $Y$.

We define a **heterogeneity function** $H_k$ for each node $k$, which satisfies:

* $H_k = 0$ when all observations in the node have the same ( Y ),
* $H_k$ is maximal when $Y$ values are uniformly mixed.

The goal is to **minimize the total impurity**:

$$
H_\text{left} + H_\text{right}
$$

Different impurity measures are used depending on the type of $Y$.

### Case 1: $Y$ Quantitative (Regression)

For regression, heterogeneity is measured by **variance**:

$$
H_k = \frac{1}{|k|}\sum_{i \in k} (y_i - \bar{y}_k)^2
$$

where:

* $|k|$ = number of observations in node $k$,
* $\bar{y}_k$ = mean of $Y$ values in node $k$.

👉 The best split is the one that **minimizes the sum of variances** of the two resulting branches:

$$
H_\text{left} + H_\text{right}
$$

Intuitively, we’re finding the split that makes each branch’s $Y$ values as similar as possible (i.e. the most homogeneous).


### Case 2: $Y$ Qualitative (Classification)

When $Y$ is categorical, we use impurity indices like **Gini** (default in scikit-learn):

$$
H_k = \sum_{i=1}^{m} p_k^i (1 - p_k^i)
$$

where $p_k^i$ is the proportion of class $i$ within node $k$.

* If all samples in a node belong to one class 👉 $H_k = 0$
* If all classes are evenly mixed 👉 $H_k$ is maximal

Again, the best split is the one that minimizes:

$$
H_\text{left} + H_\text{right}
$$

## Stopping Criteria ⛔

A node becomes **terminal** (a leaf) when:

* It’s *pure* (all observations share the same $Y$),
* No further admissible split exists,
* The number of samples is below a minimum threshold (`min_samples_split` or `min_samples_leaf`),
* The tree reaches a predefined depth (`max_depth`).

These constraints are key to preventing overfitting.

## Overfitting and Regularization 🚨

A fully grown tree often **overfits**:

* It learns every tiny detail and noise from the training data.
* It performs very well on training data but generalizes poorly to new examples.

The **bias–variance trade-off** applies here:

* A shallow tree 👉 high bias, low variance
* A deep tree 👉 low bias, high variance

The solution is to control the **tree complexity** with stopping criteria or hyperparameters like:

* `max_depth`
* `min_samples_split`
* `min_samples_leaf`

You can tune these parameters using **Grid Search** or **Cross-Validation** to find the best generalization performance.


## 💬 General Remarks & Best Practices

* **Categorical variables:** When one-hot encoded, the importance of each category is diluted across several dummy variables. Consider merging rare or similar categories before training.
* **No distribution assumptions:** Trees don’t require normalized or Gaussian features — they work directly on raw scales.
* **Implicit feature selection:** Only features that improve splits are used, which makes trees naturally selective — though potentially ignoring weakly informative features.
* **Robustness:** Because splits depend only on relative feature orderings, trees handle outliers and skewed distributions relatively well.
* **Sequential dependency:** Early bad splits can propagate errors downstream — trees are greedy and may miss global optima.
* **Regression output:** Predictions are *piecewise constant* (a “stair-step” function) rather than smooth.
* **Smooth functions:** If the true relationship between $X$ and $Y$ is continuous (e.g. linear, polynomial), a single tree will struggle to capture it smoothly — motivating ensemble methods like **Random Forests**.


## Resources 📚📚

- [Decision Trees](https://scikit-learn.org/stable/modules/tree.html)
- [Decision and Classification Trees, Clearly Explained!!!](https://www.youtube.com/watch?v=_L39rN6gz7Y)